# B.4 任務完成後怎麼停止？

# B：從結構化回答到工具使用

前置：第7章對話與8.5結構化輸出。模型提出要求，外層程式驗證後調用白名單純函數，再把結果給模型。這裏不執行模型產生的任意Python或外部請求。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：工單先驗證，再交給真正工具做**

模型文字不等於已執行，result來自白名單純函式。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/tools.svg")))

**先想一想：**只有次數上限，沒有完成信號，會知道任務成功了嗎？

工具循環要有完成信號與最大步數，像每張工單都有關閉方式。無限循環可能重複做同一件事；錯誤與未完成是不同狀態。

**把直覺寫成數學：**done、step_limit、invalid_request、needs_more_model_output各自記錄。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.retrieval import tool_loop

requests = [{"name": "add", "arguments": {"a": 2, "b": 3}}, {"done": True, "answer": 5}]
print(tool_loop(requests, max_steps=3))
print(tool_loop(requests, max_steps=1))
assert tool_loop(requests)["status"] == "done"

**如何讀結果：**人工requests只驗證外層協議；真實模型每一步產生新請求，評成功率與實際調用次數。

**只改一件事：**只刪除done，觀察狀態不應被寫成成功。
